# Population Planner vs Baseline

This notebook is for the clean `feature/population-planner` branch.

Main action cells:
1. `Run Baseline`
2. `Run planner_population`
3. `Compare Logs and Plots`


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys

import matplotlib.pyplot as plt
import pandas as pd

from hpc_llm_setup import (
    config_from_env,
    ensure_eoh_src_on_path,
    resolve_model_id,
    start_hpc_bridge,
    stop_hpc_bridge,
    test_bridge,
)

project_root, eoh_src = ensure_eoh_src_on_path(Path.cwd())
notebooks_dir = project_root / 'notebooks'
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from eoh import eoh
from eoh.utils.getParas import Paras

RUN_ROOT = project_root / 'compare_runs' / 'population_planner_vs_baseline'
BASELINE_OUT = RUN_ROOT / 'baseline'
PLANNER_OUT = RUN_ROOT / 'planner_population'

cfg = config_from_env()
bridge_server = None
bridge_thread = None
bridge_url = None
model_id = None


def ensure_bridge():
    global bridge_server, bridge_thread, bridge_url, model_id
    if bridge_url is not None:
        return bridge_url, model_id
    try:
        bridge_server, bridge_thread, bridge_url, model_id = start_hpc_bridge(cfg)
        print(f'Bridge started at {bridge_url}')
    except OSError:
        bridge_url = f'http://127.0.0.1:{cfg.port}/completions'
        model_id = resolve_model_id(cfg)
        print(f'Reusing existing bridge at {bridge_url}')
    status, payload = test_bridge(bridge_url)
    print('Bridge test:', status, payload)
    return bridge_url, model_id


def run_mode(mode: str, output_path: Path, n_pop: int, pop_size: int = 8, n_proc: int = 8, eval_instances: int = 256, holdout_instances: int = 64, reset: bool = True):
    bridge_url, model_id = ensure_bridge()
    output_path = Path(output_path)
    if reset and output_path.exists():
        shutil.rmtree(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    paras = Paras()
    paras.set_paras(
        method='eoh',
        problem='bp_online',
        ec_operators=['e1', 'e2', 'm1', 'm2', 'm3'],
        llm_use_local=True,
        llm_local_url=bridge_url,
        llm_model=model_id,
        ec_pop_size=pop_size,
        ec_n_pop=n_pop,
        exp_n_proc=n_proc,
        exp_output_path=str(output_path),
        exp_debug_mode=False,
        eval_parallel_instances=max(1, min(8, n_proc)),
        eval_instances_per_gen=eval_instances,
        holdout_instances=holdout_instances,
        holdout_eval_interval=1,
        eoh_mode=mode,
        log_full_population=True,
    )
    runner = eoh.EVOL(paras)
    runner.run()
    return output_path


def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def load_mode_logs(mode_path: Path):
    mode_path = Path(mode_path)
    return {
        'run': pd.DataFrame(read_jsonl(mode_path / 'results' / 'run_log.jsonl')),
        'operator': pd.DataFrame(read_jsonl(mode_path / 'results' / 'operator_events.jsonl')),
        'cards': pd.DataFrame(read_jsonl(mode_path / 'results' / 'heuristic_cards.jsonl')),
        'summary': pd.DataFrame(read_jsonl(mode_path / 'results' / 'population_summary.jsonl')),
        'planner': pd.DataFrame(read_jsonl(mode_path / 'results' / 'population_planner_output.jsonl')),
        'executed': pd.DataFrame(read_jsonl(mode_path / 'results' / 'executed_interventions.jsonl')),
        'lineage': pd.DataFrame(read_jsonl(mode_path / 'results' / 'offspring_lineage.jsonl')),
    }

print('Project root:', project_root)
print('Run root:', RUN_ROOT)


In [ ]:
# Run Baseline
run_mode(
    mode='baseline',
    output_path=BASELINE_OUT,
    n_pop=10,
    pop_size=8,
    n_proc=8,
    eval_instances=256,
    holdout_instances=64,
    reset=True,
)


In [ ]:
# Run planner_population
run_mode(
    mode='planner_population',
    output_path=PLANNER_OUT,
    n_pop=10,
    pop_size=8,
    n_proc=8,
    eval_instances=256,
    holdout_instances=64,
    reset=True,
)


In [ ]:
# Compare Logs and Plots
baseline_logs = load_mode_logs(BASELINE_OUT)
planner_logs = load_mode_logs(PLANNER_OUT)

baseline_df = baseline_logs['run']
planner_df = planner_logs['run']
executed_df = planner_logs['executed']
planner_out_df = planner_logs['planner']
lineage_df = planner_logs['lineage']

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

if not baseline_df.empty:
    y = baseline_df['train_fitness'] if 'train_fitness' in baseline_df.columns else baseline_df['best_fitness']
    axes[0].plot(baseline_df['gen'], y, label='baseline')
if not planner_df.empty:
    y = planner_df['train_fitness'] if 'train_fitness' in planner_df.columns else planner_df['best_fitness']
    axes[0].plot(planner_df['gen'], y, label='planner_population')
axes[0].set_title('Train Fitness vs Generation')
axes[0].set_xlabel('gen')
axes[0].set_ylabel('fitness (lower better)')
axes[0].legend()

if not baseline_df.empty:
    axes[1].plot(baseline_df['gen'], baseline_df['invalid_rate'], label='baseline')
if not planner_df.empty:
    axes[1].plot(planner_df['gen'], planner_df['invalid_rate'], label='planner_population')
axes[1].set_title('Invalid Rate vs Generation')
axes[1].set_xlabel('gen')
axes[1].set_ylabel('invalid_rate')
axes[1].legend()

if not executed_df.empty:
    mode_counts = executed_df['intervention'].apply(lambda x: x.get('execution_mode') if isinstance(x, dict) else None).dropna().value_counts()
    axes[2].bar(mode_counts.index.astype(str), mode_counts.values)
axes[2].set_title('planner_population Executed Modes')
axes[2].set_xlabel('execution_mode')
axes[2].set_ylabel('count')

plt.tight_layout()
plt.show()

summary_rows = []
if not baseline_df.empty:
    summary_rows.append({
        'mode': 'baseline',
        'best_final_fitness': float((baseline_df['train_fitness'] if 'train_fitness' in baseline_df.columns else baseline_df['best_fitness']).iloc[-1]),
        'best_seen_fitness': float((baseline_df['train_fitness'] if 'train_fitness' in baseline_df.columns else baseline_df['best_fitness']).min()),
        'invalid_rate_mean': float(baseline_df['invalid_rate'].mean()),
    })
if not planner_df.empty:
    summary_rows.append({
        'mode': 'planner_population',
        'best_final_fitness': float((planner_df['train_fitness'] if 'train_fitness' in planner_df.columns else planner_df['best_fitness']).iloc[-1]),
        'best_seen_fitness': float((planner_df['train_fitness'] if 'train_fitness' in planner_df.columns else planner_df['best_fitness']).min()),
        'invalid_rate_mean': float(planner_df['invalid_rate'].mean()),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

if not planner_out_df.empty:
    planner_view = planner_out_df[['gen', 'planner_output']].copy()
    planner_view['population_assessment'] = planner_view['planner_output'].apply(lambda x: x.get('population_assessment') if isinstance(x, dict) else None)
    planner_view['overall_strategy'] = planner_view['planner_output'].apply(lambda x: x.get('overall_strategy') if isinstance(x, dict) else None)
    planner_view['intervention_count'] = planner_view['planner_output'].apply(lambda x: len(x.get('interventions', [])) if isinstance(x, dict) else 0)
    display(planner_view[['gen', 'population_assessment', 'overall_strategy', 'intervention_count']].tail(10))

if not lineage_df.empty:
    lineage_cols = [
        col for col in [
            'gen',
            'offspring_id',
            'execution_mode',
            'target_ids',
            'fitness_delta',
            'fragmentation_delta',
            'resource_opening_rate_early_delta',
            'order_sensitivity_delta',
        ] if col in lineage_df.columns
    ]
    display(lineage_df[lineage_cols].tail(20))
